# Exact 0.814 Optuna XGB Branch Reproduction

This demo notebook follows the training/submission cells from `0-814-optuna-xgb-space-titanic.ipynb` and removes the EDA-only cells. The only intentional changes are portable local paths and compatibility with current `scikit-learn`.

Important: the original notebook did not set random seeds for `shuffle`, `KMeansSMOTE`, or the final `XGBClassifier`, so exact row-level CSV reproduction can vary. We keep a reference highest-score CSV and compare against it at the end.

In [ ]:
# Install the exact project dependencies into the active VSCode/Jupyter kernel.
# This avoids the common "No module named imblearn" issue caused by using a different interpreter.
from pathlib import Path
import subprocess
import sys


def find_project_root():
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "requirements.txt").exists():
            return candidate.resolve()
    raise FileNotFoundError("Open this notebook from the repository folder or its notebooks/ folder.")


PROJECT_ROOT = find_project_root()
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "requirements.txt")])
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from imblearn.over_sampling import KMeansSMOTE
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils import shuffle
import xgboost as xgb
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

In [ ]:
def find_data_dir():
    candidates = [
        PROJECT_ROOT / "data",
        PROJECT_ROOT,
        Path.cwd() / "data",
        Path.cwd(),
        Path.cwd().parent / "data",
        Path.cwd().parent,
    ]
    for path in candidates:
        if all((path / name).exists() for name in ["train.csv", "test.csv", "sample_submission.csv"]):
            return path.resolve()
    checked = "\n".join(f"- {path}" for path in candidates)
    raise FileNotFoundError(f"Could not find train.csv, test.csv, sample_submission.csv. Checked:\n{checked}")


def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


DATA_DIR = find_data_dir()
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
Submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print(f"Data directory: {DATA_DIR}")
print(f"train: {train.shape}, test: {test.shape}, sample_submission: {Submission.shape}")
display(train.head())

In [ ]:
def get_score(model, X, y):
    n = cross_val_score(model, X, y, scoring="accuracy", cv=10)
    return n


params_XGB_best = {
    "lambda": 3.06,
    "alpha": 4.582,
    "colsample_bytree": 0.93,
    "subsample": 0.96,
    "n_estimators": 725,
    "max_depth": 5,
    "learning_rate": 0.05,
}
params_XGB_best

## Original Feature Engineering Cells

The next cells mirror the original notebook sequence: combine train/test, zero expenses for CryoSleep passengers, infer missing grouped values by room, split cabin/name, then one-hot encode the compact feature set.

In [ ]:
train_test = train._append(test, ignore_index=True)

Expenses_columns = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
train_test.loc[:, Expenses_columns] = train_test.apply(lambda x: 0 if x.CryoSleep == True else x, axis=1)
train_test["Expenses"] = train_test.loc[:, Expenses_columns].sum(axis=1)
train_test.loc[:, ["CryoSleep"]] = train_test.apply(
    lambda x: True if x.Expenses == 0 and pd.isna(x.CryoSleep) else x,
    axis=1,
)

train_test.Name = train_test.Name.fillna("Unknown Unknown")
train_test.loc[:, ["Room"]] = train_test.PassengerId.apply(lambda x: x[0:4])

print(train_test.shape)
display(train_test[["PassengerId", "CryoSleep", "Expenses", "Room"]].head())

In [ ]:
guide_VIP = train_test.loc[:, ["Room", "VIP"]].dropna().drop_duplicates("Room")
guide_Cabin = train_test.loc[:, ["Room", "Cabin"]].dropna().drop_duplicates("Room")
guide_HomePlanet = train_test.loc[:, ["Room", "HomePlanet"]].dropna().drop_duplicates("Room")
guide_Destination = train_test.loc[:, ["Room", "Destination"]].dropna().drop_duplicates("Room")

train_test = pd.merge(train_test, guide_Cabin, how="left", on="Room", suffixes=("", "_y"))
train_test = pd.merge(train_test, guide_VIP, how="left", on="Room", suffixes=("", "_y"))
train_test = pd.merge(train_test, guide_HomePlanet, how="left", on="Room", suffixes=("", "_y"))
train_test = pd.merge(train_test, guide_Destination, how="left", on="Room", suffixes=("", "_y"))

train_test.loc[:, ["VIP"]] = train_test.apply(lambda x: x.VIP_y if pd.isna(x.VIP) else x, axis=1)
train_test.loc[:, ["Cabin"]] = train_test.apply(lambda x: x.Cabin_y if pd.isna(x.Cabin) else x, axis=1)
train_test.loc[:, ["HomePlanet"]] = train_test.apply(lambda x: x.HomePlanet_y if pd.isna(x.HomePlanet) else x, axis=1)
train_test.loc[:, ["Destination"]] = train_test.apply(lambda x: x.Destination_y if pd.isna(x.Destination) else x, axis=1)

print(train_test[["VIP", "Cabin", "HomePlanet", "Destination"]].isna().sum())
display(train_test[["Room", "VIP", "Cabin", "HomePlanet", "Destination"]].head())

In [ ]:
train_test.loc[:, ["Cabin_1"]] = train_test.Cabin.str.split("/", expand=True).iloc[:, 0]
train_test.loc[:, ["Cabin_2"]] = train_test.Cabin.str.split("/", expand=True).iloc[:, 1]
train_test.loc[:, ["Cabin_3"]] = train_test.Cabin.str.split("/", expand=True).iloc[:, 2]

train_test.loc[:, ["FirstName"]] = train_test.Name.str.split(" ", expand=True).iloc[:, 0]
train_test.loc[:, ["SecondName"]] = train_test.Name.str.split(" ", expand=True).iloc[:, 1]
train_test["Name_key"] = train_test["SecondName"] + train_test["Room"]

display(train_test[["Cabin", "Cabin_1", "Cabin_2", "Cabin_3", "Name_key"]].head())

In [ ]:
num_cols = ["ShoppingMall", "FoodCourt", "RoomService", "Spa", "VRDeck", "Expenses", "Age"]
cat_cols = ["CryoSleep", "Cabin_1", "Cabin_3", "VIP", "HomePlanet", "Destination"]
transported = ["Transported"]
train_test = train_test[num_cols + cat_cols + transported].copy()

num_imp = SimpleImputer(strategy="mean")
cat_imp = SimpleImputer(strategy="most_frequent")
ohe = make_one_hot_encoder()

train_test[num_cols] = pd.DataFrame(num_imp.fit_transform(train_test[num_cols]), columns=num_cols)
train_test[cat_cols] = pd.DataFrame(cat_imp.fit_transform(train_test[cat_cols]), columns=cat_cols)
temp_train = pd.DataFrame(ohe.fit_transform(train_test[cat_cols]), columns=ohe.get_feature_names_out())
train_test = train_test.drop(cat_cols, axis=1)
train_test = pd.concat([train_test, temp_train], axis=1)

print(train_test.shape)
display(train_test.head())

In [ ]:
train_model = train_test[train_test["Transported"].notnull()].copy()
train_model.Transported = train_model.Transported.astype("int")
test_model = train_test[train_test["Transported"].isnull()].drop("Transported", axis=1)

X = train_model.drop("Transported", axis=1)
y = train_model.Transported

print(X.shape, y.shape, test_model.shape)
display(X.head())

## Original Model Branch

These are the original modeling cells: unseeded shuffle, 10-fold CV, IsolationForest diagnostic only, manual feature drop list, KMeansSMOTE, final XGBoost fit, and CSV export.

In [ ]:
X, y = shuffle(X, y)
X = X.reset_index(drop=True)
y = y.reset_index(drop=True)

cv_after_shuffle = get_score(xgb.XGBClassifier(**params_XGB_best), X, y).mean()
print(cv_after_shuffle)

In [ ]:
features_isolation = ["ShoppingMall", "FoodCourt", "RoomService", "Spa", "VRDeck", "Age"]

isf = IsolationForest(n_jobs=-1, random_state=1, n_estimators=100, contamination=0.003)
isf.fit(X[features_isolation], y)

rows = pd.DataFrame(isf.predict(X[features_isolation]), columns=["feature"])
rows_ind = rows[rows.feature == 1]
X_1 = X.iloc[rows_ind.index].reset_index(drop=True)
y_1 = y.iloc[rows_ind.index].reset_index(drop=True)

cv_isolation_diagnostic = get_score(xgb.XGBClassifier(**params_XGB_best), X_1, y_1).mean()
print(cv_isolation_diagnostic)
print(f"Rows kept by diagnostic branch: {len(X_1)}")

In [ ]:
drop_list = [
    "ShoppingMall",
    "Age",
    "CryoSleep_True",
    "HomePlanet_Earth",
    "HomePlanet_Europa",
    "VIP_True",
    "HomePlanet_Mars",
    "Destination_PSO J318.5-22",
    "VIP_False",
    "Destination_55 Cancri e",
    "FoodCourt",
    "Destination_TRAPPIST-1e",
]

X = X.drop(drop_list, axis=1)
test_model = test_model.drop(drop_list, axis=1)

cv_after_drop = get_score(xgb.XGBClassifier(**params_XGB_best), X, y).mean()
print(cv_after_drop)
print(X.shape, test_model.shape)

In [ ]:
sm = KMeansSMOTE(sampling_strategy=1, n_jobs=-1)
X_sm, y_sm = sm.fit_resample(X, y)
X = X_sm
y = y_sm

y.value_counts().sort_index()

In [ ]:
pred_XGB = xgb.XGBClassifier(**params_XGB_best).fit(X, y).predict(test_model)
pred_XGB[:20]

In [ ]:
Submission["Transported"] = pred_XGB
Submission["Transported"] = Submission["Transported"] > 0.5

output_path = PROJECT_ROOT / "submissions" / "Submission_XGB_exact.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
Submission.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"True count: {int(Submission['Transported'].sum())}")
print(f"True rate: {Submission['Transported'].mean():.6f}")
display(Submission)

In [ ]:
reference_path = PROJECT_ROOT / "submissions" / "submission_xgb_reference_0814.csv"
metrics_path = PROJECT_ROOT / "experiments" / "tables" / "notebook_0814_exact_run_metrics.json"
metrics_path.parent.mkdir(parents=True, exist_ok=True)

metrics = {
    "source_notebook": "0-814-optuna-xgb-space-titanic.ipynb",
    "note": "Original notebook leaves shuffle, KMeansSMOTE, and XGB estimator randomness unseeded.",
    "cv_after_shuffle": float(cv_after_shuffle),
    "cv_isolation_diagnostic": float(cv_isolation_diagnostic),
    "cv_after_drop": float(cv_after_drop),
    "smote_class_counts": {str(k): int(v) for k, v in y.value_counts().sort_index().to_dict().items()},
    "output_true_count": int(Submission["Transported"].sum()),
    "output_true_rate": float(Submission["Transported"].mean()),
}

if reference_path.exists():
    reference = pd.read_csv(reference_path)
    diff_count = int((Submission["Transported"].astype(bool) != reference["Transported"].astype(bool)).sum())
    metrics["reference_comparison"] = {
        "reference_csv": reference_path.relative_to(PROJECT_ROOT).as_posix(),
        "row_diff_count": diff_count,
        "reference_true_count": int(reference["Transported"].astype(bool).sum()),
        "reference_true_rate": float(reference["Transported"].astype(bool).mean()),
    }
    print(f"Reference highest-score CSV found: {reference_path.name}")
    print(f"Row-level difference vs this run: {diff_count}")
else:
    metrics["reference_comparison"] = "Reference CSV not present."
    print("Reference highest-score CSV not present; generated submission is still saved.")

metrics_path.write_text(__import__("json").dumps(metrics, indent=2), encoding="utf-8")
print(f"Saved metrics: {metrics_path}")
metrics